In [1]:
!pip install pedalboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 71.1 MB/s eta 0:00:00:00:010:01


In [2]:
!pip install albumentations

In [3]:
pip install --upgrade soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.5 MB/s eta 0:00:0000:01
  Attempting uninstall: soundfile
    Found existing installation: soundfile 0.12.1
    Uninstalling soundfile-0.12.1:
      Successfully uninstalled soundfile-0.12.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install torch-audiomentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00
  Created wheel for julius: filename=julius-0.2.7-py3-none-any.whl size=21869 sha256=220482039435c25411d5b8dbae8337ad5b706e49bb05a85768f919b9b31f648e
  Stored in directory: /root/.cache/pip/wheels/b9/b2/05/f883527ffcb7f2ead5438a2c23439aa0c881eaa9a4c80256f4
Successfully built julius


In [5]:
!pip install audiomentations

INFO: pip is looking at multiple versions of numpy-minmax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of numpy-rms to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 6.2 MB/s eta 0:00:00


In [30]:
pip install colorednoise

Note: you may need to restart the kernel to use updated packages.


In [52]:
import os
import gc
import glob
import logging
import warnings
import math
from pathlib import Path
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
import librosa
import soundfile
import IPython.display as ipd
# import webrtcvad
from tqdm import tqdm
import albumentations as A

from audiomentations import Compose as amCompose
from audiomentations import OneOf as amOneOf
from audiomentations import AddBackgroundNoise, PolarityInversion, Gain, GainTransition, TimeStretch

from torch_audiomentations import Compose
from torch_audiomentations import PitchShift, Shift, OneOf, AddColoredNoise

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

import pedalboard
from pedalboard import Reverb

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
import torchaudio
from torchaudio.functional import add_noise
import colorednoise as cn # Pink noise용
from torch.distributions import Beta

In [ ]:
train = pd.read_csv('/kaggle/input/birdclef-2025/train.csv')

In [ ]:
submission = pd.read_csv('/kaggle/input/birdclef-2025/sample_submission.csv')

In [ ]:
train_audio_path = '/kaggle/input/birdclef-2025/train_audio'
folders = [f for f in os.listdir(train_audio_path) if os.path.isdir(os.path.join(train_audio_path, f))]

In [ ]:
# for folder in folders:
#     print(folder)
#     folder_path = os.path.join(train_audio_path, folder)
#     files = os.listdir(folder_path)
#     print(len(files))

#     for file in files:
#         file_path = os.path.join(folder_path, file)
#         y, sr = librosa.load(file_path, sr=32000)
#         duration = librosa.get_duration(y=y, sr=sr)
#         if duration < 3: 
#             print(file)
            
#     print('=' * 10)

In [ ]:
# for folder in folders:
#     folder_path = os.path.join(train_audio_path, folder)
#     files = os.listdir(folder_path)

#     if len(files) > 100:
#         print(folder)
#         print(len(files))
#         print('=' * 10)

In [ ]:
file_path = 'linwoo1/XC63209.ogg' # 'thbeup1/XC428295.ogg' XC355995

audio_path = os.path.join(train_audio_path, file_path)
print(audio_path)
ipd.Audio(audio_path, rate=32000)

In [ ]:
target_columns = submission.columns.tolist()
num_classes = len(target_columns[1:])
num_classes

In [ ]:
train['primary_label'].value_counts()

In [ ]:
train['secondary_labels'].value_counts()

In [ ]:
train['secondary_labels'] # 문자열(object)

## Bandpass Filtering

In [ ]:
# audio_path = '/kaggle/input/birdclef-2025/train_audio/1346504/CSA18803.ogg' # '/kaggle/input/birdclef-2025/train_audio/1194042/CSA18783.ogg' # '/kaggle/input/birdclef-2025/train_audio/476538/XC926710.ogg' # '/kaggle/input/birdclef-2025/train_audio/1139490/CSA36389.ogg'
audio_path = '/kaggle/input/birdclef-2025/train_audio/22976/iNat134883.ogg' # 전체 무음 파일
# audio_path = '/kaggle/input/birdclef-2025/train_audio/135045/iNat1207345.ogg'

fs = 32000
y, sr = librosa.load(audio_path, sr=fs)

def butter_bandstop(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='bandstop')
    return b, a

def butter_bandstop_filter(y, lowcut, highcut, fs, order=5):
    b, a = butter_bandstop(lowcut, highcut, fs, order=order)
    return lfilter(b, a, y)

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(y, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, y)
    return y

In [2]:
# original_sound_file_path = list(Path(Config().train_audio + '/whbman1/').glob('*.ogg'))[0]

In [3]:
# fs = 32000
# y, sr = librosa.load(original_sound_file_path, sr=fs)

In [4]:
# threshold = 0.2
# indices = np.where(np.abs(y) > threshold)[0]

# if len(indices) > 0:
#     end_idx = indices[0]
# else:
#     end_idx = len(y)
    
# y_filtered_r = y[:end_idx]

In [5]:
# y_filtered_r.shape # (56384,)

## Analytical Base - (Amplitude, 진폭)

- 음성은 **시간**, **주파수**, **진폭**(**Amplitude**)로 구성되어있음.

In [20]:
plt.figure(figsize=(15, 7))

plt.subplot(2, 1, 1)
plt.plot(np.linspace(0, len(y)/fs, len(y)), y)
plt.title('Time Domain Signal')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')

plt.subplot(2, 1, 2)
D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
librosa.display.specshow(D, sr=fs, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')

plt.tight_layout()
plt.show()

In [ ]:
# D = librosa.stft(y) # STFT 변환
# freqs = librosa.fft_frequencies(sr=fs) # STFT 주파수 계산

# plt.figure(figsize=(10, 4))
# librosa.display.specshow(librosa.amplitude_to_db(np.abs(D), ref=np.max),
#                         y_axis='log', x_axis='time', sr=fs)

# plt.colorbar(label='dB')
# plt.title('Spectrogram')
# plt.show()

In [ ]:
# y_filtered = butter_bandpass_filter(y, lowcut=2048, highcut=8192, fs=fs)

In [ ]:
# # 평균 노이즈 감산

# D = librosa.stft(y)
# D_magnitude, D_phase = librosa.magphase(D)

# # 노이즈가 있는 구간 설정
# hop_length = 512
# frames_per_second = sr // hop_length
# noise_frames = frames_per_second * 2

# # 전체 프레임 수 계산
# total_frames = D.shape[1]

# noise_mag = np.mean(np.abs(D[:, :-noise_frames]), axis=1, keepdims=True)
# D_denoised = np.maximum(0, np.abs(D_magnitude) - noise_mag) * D_phase
# y_filtered = librosa.istft(D_denoised)

In [ ]:
# threshold = np.percentile(np.abs(y), 25)
# threshold = 0.2

# indices = np.where(np.abs(y) > threshold)[0]

# start_idx = np.where(np.abs(y) > threshold)[0][0] if np.any(np.abs(y) > threshold) else 0

# end_idx = np.where(np.abs(y) > threshold)[0][-1] if np.any(np.abs(y) > threshold) else len(y)

# y_filtered = y[start_idx:end_idx]

# if len(indices) > 0:
#     end_idx = indices[0]
# else:
#     end_idx = len(y)
    
# y_filtered = y[:end_idx]

https://www.kaggle.com/competitions/birdclef-2025/discussion/568886

In [ ]:
plt.figure(figsize=(15, 7))
chunk_len = 0.05
power = y ** 2
chunk = int(chunk_len * sr)
pad = int(np.ceil(len(power) / chunk) * chunk - len(power))
power = np.pad(power, (0, pad))
power = power.reshape((-1, chunk)).sum(axis=1)
t = np.arange(len(power))*chunk_len
plt.plot(t, 10 * np.log10(power))

In [ ]:
# torch.set_num_threads(1)
# model, (get_speech_timestamps, _, read_audio, _, _) = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')

In [ ]:
model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=True)

(get_speech_timestamps,
 _, read_audio,
 *_) = utils

In [ ]:
speech_timestamps = get_speech_timestamps(
    torch.Tensor(y), model, sampling_rate=fs, threshold=0.3,
    min_speech_duration_ms=250,
    min_silence_duration_ms=500,
    window_size_samples=1024,
    speech_pad_ms=100
)

In [ ]:
speech_timestamps

In [ ]:
# y_filtered = np.zeros_like(y)
# for st in speech_timestamps:
#     y_filtered[st['start']: st['end']] = y[st['start']: st['end']]

In [ ]:
segments = []
temp = 0

for st in speech_timestamps:
    start, end = st['start'], st['end']
    segments.append(y[temp: start])
    temp = end

segments.append(y[temp: ])

y_filtered = np.concatenate(segments)

In [ ]:
y_filtered, y_filtered.shape

In [ ]:
# speech_timestamps + 무음 구간 포함
# rms = librosa.feature.rms(y=y_filtered)[0]
# threshold = 0.01
# silent_intervals = np.where(rms > threshold)[0]

In [ ]:
# len(silent_intervals)

In [ ]:
# segments = []
# temp = 0
# for i in silent_intervals:
#     segments.append(y_filtered[temp: i])
#     temp = i

# segments.append(y_filtered[temp: ])

# y_filtered = np.concatenate(segments)

In [ ]:
# y_filtered

In [ ]:
plt.figure(figsize=(15, 7))

plt.subplot(2, 1, 1)
plt.plot(np.linspace(0, len(y_filtered)/fs, len(y_filtered)), y_filtered)
plt.title('Time Domain Signal')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')

plt.subplot(2, 1, 2)
D = librosa.amplitude_to_db(np.abs(librosa.stft(y_filtered)), ref=np.max)
librosa.display.specshow(D, sr=fs, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')

plt.tight_layout()
plt.show()

**조류 오디오와 노이즈의 음역대가 같음.**

-> 단순 필터링으로는 노이즈를 제거하기 어려움.

In [ ]:
# !pip install noisereduce

In [ ]:
# import noisereduce as nr

# noise_part = y[:fs*2]

# y_filtered = nr.reduce_noise(y=y, sr=sr, y_noise=noise_part, prop_decrease=0.8)

In [ ]:
!rm -rf /kaggle/working/* bash

In [ ]:
# soundfile.write('filtered_audio.ogg', y_filtered.astype('float32'), fs)

# **꺼짐 문제** -> 해결 -> .wav -> ogg 변환

soundfile.write('filtered_audio.wav', y_filtered.astype('float32'), fs)
!ffmpeg -i filtered_audio.wav filtered_audio.ogg

In [ ]:
ipd.Audio(audio_path, rate=fs)
# ipd.Audio(original_sound_file_path, rate=fs)

In [ ]:
ipd.Audio('./filtered_audio.ogg', rate=fs)

In [7]:
# from pydub.silence import detect_nonsile

def detect_nonsilent_custom(y, sr, min_silence_len=300, silence_thresh=-45, hop_length=512):

    """
    min_silence_len: int -> 최소 무음 길이(밀리초)
    silence_thresh: int -> 무음으로 간주할 dB 임계값
    hop_length: int -> 프레임 간 간격 (샘플 수)
    """
    
    # 1. 프레임 길이별 RMS 에너지 계산
    # RMS - 오디오 신호의 평균적인 세기를 계산 (일반적으로 음량(볼륨)이나 신호 강도를 측정)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]

    # 2. RMS 값을 dB로 변환
    # dB - 데시벨(dB) 단위로 변환하여 비교하는 것이 더 직관적이고 스케일에 더 민감해야 감지를 잘함.
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)

    # 3. 무음 임계값 이하인 부분을 감지
    silent_frames = np.where(rms_db < silence_thresh)[0]

    # 4. 무음 프레임을 시간으로 변환
    silent_times = silent_frames * hop_length / sr * 1000 # 밀리초(ms) 단위 변환

    # 5. 연속된 무음 구간 Find
    silent_intervals = []
    if len(silent_times) > 0:
        start = silent_times[0]
        for i in range(1, len(silent_times)):
            if silent_times[i] > silent_times[i - 1] + hop_length / sr * 1000:
                silent_intervals.append((start, silent_times[i - 1]))
                start = silent_times[i]
        silent_intervals.append((start, silent_times[-1]))

    # 6. 무음이 아닌 구간
    nonsilent_intervals = []
    prev_end = 0
    for start, end in silent_intervals:
        if start - prev_end >= min_silence_len:
            nonsilent_intervals.append((prev_end, start))
        prev_end = end
    if prev_end < len(y) / sr * 1000:
        nonsilent_intervals.append((prev_end, len(y) / sr * 1000))

    return nonsilent_intervals

In [21]:
nonsilent_ranges = detect_nonsilent_custom(y_filtered, sr, min_silence_len=300, silence_thresh=-45)
print(nonsilent_ranges)

y_non_silent = np.concatenate([
    y_filtered[int(start * sr / 1000):int(end * sr / 1000)]
    for start, end in nonsilent_ranges
])

print(len(y_filtered), '->', len(y_non_silent))

In [ ]:
y_non_silent, y_non_silent.shape

In [ ]:
# soundfile.write('silent_audio.ogg', y_non_silent, fs)

# **꺼짐 문제** -> 해결 -> .wav -> ogg 변환

soundfile.write('silent_audio.wav', y_non_silent.astype('float32'), fs)
!ffmpeg -i silent_audio.wav silent_audio.ogg

In [ ]:
ipd.Audio('./silent_audio.ogg', rate=fs)

예시) Audio 파일이 100개가 넘어가는 경우

문제점 -> 100개가 넘어가는 오디오파일들을 모두 쓰는데 시간적 비용이 많이 들고, 중복되는 오디오가 있을 수 있음.

해결 -> 클러스터링을 통해서 여러 다양한 군집의 대표를 추출하여 다양성을 확보

In [9]:
# MFCC + Cluster
def extract_mfcc(file_path, fs=32000, n_mfcc=13):
    y, _ = torchaudio.load(file_path)
    mfcc = torchaudio.transforms.MFCC(sample_rate=fs, n_mfcc=n_mfcc)(y)
    mean = mfcc.mean(dim=-1).squeeze().numpy()
    std = mfcc.std(dim=-1).squeeze().numpy()
    return np.concatenate([mean, std])
    # return mfcc.mean(dim=-1).squeeze().numpy()

def select_representative_cluster_kmeans(file_paths, k=10):
    features = [extract_mfcc(fp) for fp in tqdm(file_paths)]
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(features)

    reps = []
    for cluster_id in range(k):
        indices = np.where(labels == cluster_id)[0]
        cluster_features = [features[i] for i in indices]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = [np.linalg.norm(f - centroid) for f in cluster_features]
        representative_idx = indices[np.argmin(distances)]
        reps.append(file_paths[representative_idx])
    return reps

# MelSpectrogram + Cluster
def extract_melspectrogram(file_path, fs=32000, n_mels=128):
    y, _ = librosa.load(file_path, sr=fs)
    melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)
    mean = melspec_db.mean(axis=1)
    std = melspec_db.std(axis=1)
    return np.concatenate([mean, std])
    # return melspec_db.mean(axis=1)

def select_representative_cluster_kmeans(file_paths, k=10):
    features = []
    fps = []
    for fp in tqdm(file_paths):
        feature = extract_melspectrogram(fp)
        features.append(feature)
        fps.append(fp)

    features = np.array(features)
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(features)

    reps = []
    for cluster_id in range(k):
        indices = np.where(labels == cluster_id)[0]
        cluster_features = [features[i] for i in indices]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = [np.linalg.norm(f - centroid) for f in cluster_features]
        representative_idx = indices[np.argmin(distances)]
        reps.append(fps[representative_idx])
    
    return reps

In [9]:
def get_max_mel_length(file_paths, fs=32000, n_mels=128):
    max_len = 0
    for fp in tqdm(file_paths, desc='Finding max mel length'):
        y, _ = librosa.load(fp, sr=fs)
        melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
        max_len = max(max_len, melspec.shape[1])
    return max_len

def get_min_mel_length(file_paths, fs=32000, n_mels=128):
    min_len = float('inf')
    for fp in tqdm(file_paths, desc='Finding min mel length'):
        y, _ = librosa.load(fp, sr=fs)
        melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
        min_len = min(min_len, melspec.shape[1])
    return min_len

def extract_padded_melspectrogram(file_path, max_len, fs=32000, n_mels=128):
    y, _ = librosa.load(file_path, sr=fs)
    melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)

    # padding to max_len
    current_len = melspec_db.shape[1]
    if current_len < max_len:
        pad_width = max_len - current_len
        melspec_db = np.pad(melspec_db, ((0, 0), (0, pad_width)), mode='constant')
    
    return melspec_db.flatten()

def extract_trimmed_melspectrogram(file_path, trim_len, fs=32000, n_mels=128):
    y, _ = librosa.load(file_path, sr=fs)
    melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)

    # Trim to min_len
    melspec_db = melspec_db[:, :trim_len]
    return melspec_db.flatten()

def select_representative_cluster_padded(file_paths, k=10, pca_dim=50):
    max_len = get_max_mel_length(file_paths)
    features = []
    for fp in tqdm(file_paths, desc="Extracting features"):
        feature = extract_padded_melspectrogram(fp, max_len=max_len)
        features.append(feature)

    features = np.array(features)

    # PCA
    pca = PCA(n_components=pca_dim, random_state=42)
    reduced_features = pca.fit_transform(features)

    # KMeans clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(reduced_features)

    # Select representatives
    reps = []
    for cluster_id in range(k):
        indices = np.where(labels == cluster_id)[0]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = np.linalg.norm(reduced_features[indices] - centroid, axis=1)
        representative_idx = indices[np.argmin(distances)]
        reps.append(file_paths[representative_idx])
    
    return reps

def select_representative_cluster_trimmed(file_paths, k=10, pca_dim=50):
    min_len = get_min_mel_length(file_paths)
    features = []
    for fp in tqdm(file_paths, desc="Extracting trimmed features"):
        feature = extract_trimmed_melspectrogram(fp, trim_len=min_len)
        features.append(feature)

    features = np.array(features)

    # PCA
    pca = PCA(n_components=pca_dim, random_state=42)
    reduced_features = pca.fit_transform(features)

    # KMeans clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(reduced_features)

    # Select representatives
    reps = []
    for cluster_id in range(k):
        indices = np.where(labels == cluster_id)[0]
        centroid = kmeans.cluster_centers_[cluster_id]
        distances = np.linalg.norm(reduced_features[indices] - centroid, axis=1)
        representative_idx = indices[np.argmin(distances)]
        reps.append(file_paths[representative_idx])
    
    return reps

In [ ]:
folder_path = os.path.join(train_audio_path, 'linwoo1')
file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.ogg')]
reps = select_representative_cluster_trimmed(file_paths)

In [ ]:
# MelSpectrogram padded + Kmeans
reps

In [ ]:
# MelSpectrogram trimmed + Kmeans
reps

잘 들리는 순서대로 secondary_labels를 수기 했을 수도 있음. -> 분석 해보기!!!!!!!!

In [8]:
class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)

class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):
        super(ChannelGate, self).__init__()
        self.gate_channels = gate_channels
        self.mlp = nn.Sequential(
            Flatten(),
            nn.Linear(gate_channels, gate_channels//reduction_ratio),
            nn.ReLU(),
            nn.Linear(gate_channels//reduction_ratio, gate_channels),
            nn.Sigmoid()
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.maxpool = nn.AdaptiveMaxPool2d((1, 1))

    def forward(self, x):
        x_avg_pool = self.mlp(self.avgpool(x))
        x_max_pool = self.mlp(self.maxpool(x))
        attention = x_avg_pool + x_max_pool
        attention = attention.unsqueeze(2).unsqueeze(3).expand_as(x)
        return x * attention

class SpatialGate(nn.Module):
    def __init__(self):
        super(SpatialGate, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x_avg_pool = torch.mean(x, dim=1, keepdim=True).unsqueeze(1)
        x_max_pool = torch.max(x, dim=1, keepdim=True)[0].unsqueeze(1)
        attention = torch.cat((x_avg_pool, x_max_pool), dim=1)
        attention = self.conv(attention)
        return x * attention
        
class GeM(torch.nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = torch.nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    
    def forward(self, x):
        bs, ch, h, w = x.shape
        x = torch.nn.functional.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(
            1.0 / self.p)
        x = x.view(bs, ch)
        return x

# (참고) https://bo-10000.tistory.com/126
class CBAM(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16, channel_attention=True, spatial_attention=True):
        super(CBAM, self).__init__()
        self.channel_attention = channel_attention
        self.spatial_attention = spatial_attention
        if channel_attention:
            self.ChannelGate = ChannelGate(gate_channels, reduction_ratio)
        if spatial_attention:
            self.SpatialGate = SpatialGate()
    
    def forward(self, x):
        if self.channel_attention:
            x = self.ChannelGate(x)
        if self.spatial_attention:
            x = self.SpatialGate(x)
        return x

class BirdCLEFCNN(nn.Module):
    def __init__(self, config, num_classes):
        super().__init__()

        out_indices = (3, 4) # with features_only
        features_only = True
        
        self.backbone = timm.create_model(
            config.MODEL_NAME,
            pretrained=False,
            features_only=features_only,
            in_chans=config.IN_CHANNELS,
            # num_classes=num_classes,
            out_indices=out_indices,
        )

        feature_dims = self.backbone.feature_info.channels()

        self.cbams = nn.ModuleList(CBAM(dim) for dim in feature_dims)
        self.global_pools = nn.ModuleList([GeM() for _ in out_indices])
        self.mid_features = np.sum(feature_dims)
        self.neck = nn.BatchNorm1d(self.mid_features)
        self.head = torch.nn.Linear(self.mid_features, num_classes)

    def forward(self, x):
        ms = self.backbone(x)
        ms = [cbam(m) for m, cbam in zip(ms, self.cbams)]
        h = torch.cat([global_pool(m) for m, global_pool in zip(ms, self.global_pools)], dim=1)
        x = self.neck(h)
        x = self.head(x)
        return x

In [9]:
class Mixup(nn.Module):
    def __init__(self, mix_beta, mixup_prob, mixup_double):
        super(Mixup, self).__init__()
        self.beta_distribution = Beta(mix_beta, mix_beta)
        self.mixup_prob = mixup_prob
        self.mixup_double = mixup_double

    def forward(self, X, Y, weight=None):
        p = torch.rand((1,))[0]
        if p < self.mixup_prob:
            bs = X.shape[0]
            perm = torch.randperm(bs)
            p1 = torch.rand((1,))[0]

            if p1 < self.mixup_double:
                X = X + X[perm]
                Y = Y + Y[perm]
                Y = torch.clamp(Y, 0, 1)
                if weight is None:
                    return X, Y
                else:
                    weight = 0.5 * weight + 0.5 * weight[perm]
                    return X, Y, weight
            else:
                perm2 = torch.randperm(bs)
                X = X + X[perm] + X[perm2]
                Y = Y + Y[perm] + Y[perm2]
                Y = torch.clamp(Y, 0, 1)
                if weight is None:
                    return X, Y
                else:
                    weight = (1 / 3 * weight + 1 / 3 * weight[perm] + 1 / 3 * weight[perm2])
                    return X, Y, weight
        else:
            return (X, Y, weight) if weight is not None else (X, Y)

class Mixup2(nn.Module):
    def __init__(self, mix_beta, mixup2_prob):
        super(Mixup2, self).__init__()
        self.beta_distribution = Beta(mix_beta, mix_beta)
        self.mixup2_prob = mixup2_prob

    def forward(self, X, Y, weight=None):
        p = torch.rand((1,))[0]
        if p < self.mixup2_prob:
            bs = X.shape[0]
            perm = torch.randperm(bs)
            coeffs = self.beta_distribution.rsample(torch.Size((bs,))).to(X.device)

            n_dims = len(X.shape)
            if n_dims == 2:
                X = coeffs.view(-1, 1) * X + (1 - coeffs.view(-1, 1)) * X[perm]
            elif n_dims == 3:
                X = coeffs.view(-1, 1, 1) * X + (1 - coeffs.view(-1, 1, 1)) * X[perm]
            else:
                X = coeffs.view(-1, 1, 1, 1) * X + (1 - coeffs.view(-1, 1, 1, 1)) * X[perm]

            Y = coeffs.view(-1, 1) * Y + (1 - coeffs.view(-1, 1)) * Y[perm]

            if weight is None:
                return X, Y
            else:
                weight = coeffs.view(-1) * weight + (1 - coeffs.view(-1)) * weight[perm]
                return X, Y, weight
        else:
            return (X, Y, weight) if weight is not None else (X, Y)

In [32]:
class AudioTransform:
    def __init__(self, always_apply=False, p=0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError

class CustomCompose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y

class CustomOneOf:
    def __init__(self, transforms: list, p=1.0):
        self.transforms = transforms
        self.p = p

    def __call__(self, y: np.ndarray):
        if np.random.rand() < self.p:
            n_trns = len(self.transforms)
            trns_idx = np.random.choice(n_trns)
            trns = self.transforms[trns_idx]
            y = trns(y)
        return y

class GaussianNoiseSNR(AudioTransform):
    def __init__(self, always_apply=False, p=0.5, min_snr=5.0, max_snr=40.0, **kwargs):
        super().__init__(always_apply, p)

        self.min_snr = min_snr
        self.max_snr = max_snr

    def apply(self, y: np.ndarray, **params):
        snr = np.random.uniform(self.min_snr, self.max_snr)
        a_signal = np.sqrt(y**2).max()
        a_noise = a_signal / (10 ** (snr / 20))

        white_noise = np.random.randn(len(y))
        a_white = np.sqrt(white_noise**2).max()
        augmented = (y + white_noise * 1 / a_white * a_noise).astype(y.dtype)
        return augmented

class PinkNoiseSNR(AudioTransform):
    def __init__(self, always_apply=False, p=0.5, min_snr=5.0, max_snr=20.0, **kwargs):
        super().__init__(always_apply, p)

        self.min_snr = min_snr
        self.max_snr = max_snr

    def apply(self, y: np.ndarray, **params):
        snr = np.random.uniform(self.min_snr, self.max_snr)
        a_signal = np.sqrt(y**2).max()
        a_noise = a_signal / (10 ** (snr / 20))

        pink_noise = cn.powerlaw_psd_gaussian(1, len(y))
        a_pink = np.sqrt(pink_noise**2).max()
        augmented = (y + pink_noise * 1 / a_pink * a_noise).astype(y.dtype)
        return augmented

class NoiseInjection(AudioTransform):
    def __init__(self, always_apply=False, p=0.5, max_noise_level=0.5, sr=32000):
        super().__init__(always_apply, p)

        self.noise_level = (0.0, max_noise_level)
        self.sr = sr

    def apply(self, y: np.ndarray, **params):
        noise_level = np.random.uniform(*self.noise_level)
        noise = np.random.randn(len(y))
        augmented = (y + noise * noise_level).astype(y.dtype)
        return augmented

In [33]:
# https://github.com/karolpiczak/ESC-50?tab=readme-ov-file

In [65]:
class Config:
    root_path = '/kaggle/input/birdclef-2025'
    test_soundscapes = root_path + '/test_soundscapes'
    train_audio = root_path + '/train_audio'
    train_soundscapes = root_path + '/train_soundscapes'
    recording_location = root_path + '/recording_location.txt'
    submission = root_path + '/sample_submission.csv'
    taxonomy = root_path + '/taxonomy.csv'
    train = root_path + '/train.csv'

    chainsaw = glob.glob('/kaggle/input/chainsaw/*.wav')
    chirpingbird = glob.glob('/kaggle/input/chirpingbird/*.wav')
    cracklingfire = glob.glob('/kaggle/input/cracklingfire/*.wav')
    rainfall = glob.glob('/kaggle/input/rainfall/*.wav')
    thunder = glob.glob('/kaggle/input/thunder/*.wav')
    windblow = glob.glob('/kaggle/input/windblow/*.wav')

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Audio parameters
    SR = 32000 # 32kHz Resampling
    DURATION = 10

    # Mixup parameters
    MIX_BETA = 0.4
    MIXUP_PROB = 0.5
    MIXUP_DOUBLE = 0.6
    MIX_BETA2 = 0.7
    MIXUP2_PROB = 0.3

    # Mel spectrogram
    N_FFT = 2048
    WIN_LENGTH = 2048
    HOP_LENGTH = 384
    N_MELS = 128
    FMIN = 20
    FMAX = 16000 # self.config.SR//2
    POWER = 2.0

    NORMAL = 255
    
    # Train parameters
    MODEL_NAME = 'efficientnet_b0'
    IN_CHANNELS = 3
    OPTIMIZER = 'Adam'
    LR = 0.03
    WEIGHT_DECAY = 1e-6
    SCHEDULER = None
    T_MAX = 16000
    MIN_LR = 20
    EPOCHS = 5
    CRITERION = 'BCEWithLogitsLoss'
    N_FOLD = 5
    SEED = 42
    ALPHA = 1 # 0.5
    BATCH_SIZE = 128
    NUM_WORKERS = 0

    # Inference parameters
    SECONDARY_COEF = 0.5

class BirdDataset(torch.utils.data.Dataset):
    def __init__(self, df, config, transform=None, mode='train'):
        # self.df = pd.read_csv(config.train)
        self.df = df
        self.sub = pd.read_csv(config.submission)
        self.config = config
        self.train_audio = config.train_audio
        self.transform = transform
        self.mode = mode
        self.target_columns = self.sub.columns.tolist()[1:]
        self.num_classes = len(self.target_columns)
        self.bird2id = {v: i for i, v in enumerate(self.target_columns)}

        self.am_audio_transforms = amCompose(
            [
                AddBackgroundNoise(
                    config.chirpingbird,
                    min_snr_db=0,
                    max_snr_db=3,
                    # noise_transform=PolarityInversion(),
                    p=0.6,
                ),
                AddBackgroundNoise(
                    config.rainfall + config.windblow + config.thunder,
                    min_snr_db=0,
                    max_snr_db=3,
                    # noise_transform=PolarityInversion(),
                    p=0.4,
                ),
                AddBackgroundNoise(
                    config.cracklingfire + config.chainsaw, 
                    min_snr_db=0, 
                    max_snr_db=3,
                    # noise_transform=PolarityInversion(),
                    p=0.3
                ),
                Gain(min_gain_db=-12, max_gain_db=12, p=0.2),
                # OneOf(
                #     [
                #         Gain(min_gain_db=-15, max_gain_db=15, p=0.8),
                #         Gain(min_gain_db=-12, max_gain_db=12, p=0.2),
                #     ],
                #     p=1.0,
                # ),
            ]
        )
        
        self.np_audio_transforms = CustomCompose([
            CustomOneOf([
                NoiseInjection(p=0.5, max_noise_level=0.04),
                GaussianNoiseSNR(p=0.5),
                PinkNoiseSNR(p=0.5)
            ]),
        ])

        # Audio transforms
        self.audio_transforms = Compose(
            [
                # AddColoredNoise(p=0.5),
                PitchShift(
                    min_transpose_semitones=-4,
                    max_transpose_semitones=4,
                    sample_rate=self.config.SR,
                    p=0.4,
                ),
                Shift(min_shift=-0.5, max_shift=0.5, p=0.4),
            ]
        )

        self.time_mask_transform = torchaudio.transforms.TimeMasking(
            time_mask_param=60, iid_masks=True, p=0.5
        )
        
        self.freq_mask_transform = torchaudio.transforms.FrequencyMasking(
            freq_mask_param=24, iid_masks=True
        )

        self.mixup = Mixup(
            mix_beta=self.config.MIX_BETA,
            mixup_prob=self.config.MIXUP_PROB,
            mixup_double=self.config.MIXUP_DOUBLE
        )
        
        self.mixup2 = Mixup2(
            mix_beta=self.config.MIX_BETA2, 
            mixup2_prob=self.config.MIXUP2_PROB
        )

        # Mel Spectrogram transforms
        self.melspec_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.config.SR,
            n_fft=self.config.N_FFT,
            win_length=self.config.WIN_LENGTH,
            hop_length=self.config.HOP_LENGTH,
            n_mels=self.config.N_MELS,
            f_min=self.config.FMIN,
            f_max=self.config.FMAX,
            power=self.config.POWER,
            center=True,
            pad_mode='constant', #'reflect',
            norm='slaney',
            onesided=True,
            mel_scale='slaney' #'htk',
        )

        if self.config.DEVICE == 'cuda':
            self.melspec_transform = self.melspec_transform.cuda()
        else:
            self.melspec_transform = self.melspec_transform.cpu()

        self.db_transform = torchaudio.transforms.AmplitudeToDB(
            top_db=80 #  style='power', 
        )            

    def __len__(self):
        return len(self.df)

    def process_audio(self, y, fs, model, utils):

        # 1. Bandpass Filtering - 사람 음성 주파수 구간만 제외 X
        
        # 2. 분석적 기반, Analytical base - (Amplitude, 진폭) X -> 일부 음성은 감지하지 못함.
        ###################################################
        # threshold = 0.2
        # indices = np.where(np.abs(y) > threshold)[0]
        
        # if len(indices) > 0:
        #     end_idx = indices[0]
        # else:
        #     end_idx = len(y)
            
        # y_filtered = y[:end_idx]
        
        # 3. 사전 학습된 모델 사용 (Voice Activity Detection, VAD) -> 대체적으로 정확히 감지함.
        # torch.set_num_threads(1)
        # model, (get_speech_timestamps, _, read_audio, _, _) = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')

        # model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
        #                       model='silero_vad',
        #                       force_reload=True)

        (get_speech_timestamps,
         _, read_audio,
         *_) = utils
        
        speech_timestamps = get_speech_timestamps(
            torch.Tensor(y), model, sampling_rate=fs, threshold=0.3,
            min_speech_duration_ms=250,
            min_silence_duration_ms=500,
            window_size_samples=1024,
            speech_pad_ms=100
        )

        segments = []
        temp = 0
        
        for st in speech_timestamps:
            start, end = st['start'], st['end']
            segments.append(y[temp: start])
            temp = end
        
        segments.append(y[temp: ])
        
        y_filtered = np.concatenate(segments)

        return y_filtered

    def detect_nonsilent_custom(self, y, sr, min_silence_len=300, silence_thresh=-45, hop_length=512):

        """
        min_silence_len: int -> 최소 무음 길이(밀리초)
        silence_thresh: int -> 무음으로 간주할 dB 임계값
        hop_length: int -> 프레임 간 간격 (샘플 수)
        """
        
        # 1. 프레임 길이별 RMS 에너지 계산
        # RMS - 오디오 신호의 평균적인 세기를 계산 (일반적으로 음량(볼륨)이나 신호 강도를 측정)
        rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]
    
        # 2. RMS 값을 dB로 변환
        # dB - 데시벨(dB) 단위로 변환하여 비교하는 것이 더 직관적이고 스케일에 더 민감해야 감지를 잘함.
        rms_db = librosa.amplitude_to_db(rms, ref=np.max)
    
        # 3. 무음 임계값 이하인 부분을 감지
        silent_frames = np.where(rms_db < silence_thresh)[0]
    
        # 4. 무음 프레임을 시간으로 변환
        silent_times = silent_frames * hop_length / sr * 1000 # 밀리초(ms) 단위 변환
    
        # 5. 연속된 무음 구간 Find
        silent_intervals = []
        if len(silent_times) > 0:
            start = silent_times[0]
            for i in range(1, len(silent_times)):
                if silent_times[i] > silent_times[i - 1] + hop_length / sr * 1000:
                    silent_intervals.append((start, silent_times[i - 1]))
                    start = silent_times[i]
            silent_intervals.append((start, silent_times[-1]))
    
        # 6. 무음이 아닌 구간
        nonsilent_intervals = []
        prev_end = 0
        for start, end in silent_intervals:
            if start - prev_end >= min_silence_len:
                nonsilent_intervals.append((prev_end, start))
            prev_end = end
        if prev_end < len(y) / sr * 1000:
            nonsilent_intervals.append((prev_end, len(y) / sr * 1000))
    
        return nonsilent_intervals

    def get_max_mel_length(self, file_paths, fs=32000, n_mels=128):
        max_len = 0
        for fp in tqdm(file_paths, desc='Finding max mel length'):
            y, _ = librosa.load(fp, sr=fs)
            melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
            max_len = max(max_len, melspec.shape[1])
        return max_len

    def get_min_mel_length(self, file_paths, fs=32000, n_mels=128):
        min_len = float('inf')
        for fp in tqdm(file_paths, desc='Finding min mel length'):
            y, _ = librosa.load(fp, sr=fs)
            melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
            min_len = min(min_len, melspec.shape[1])
        return min_len
    
    def extract_padded_melspectrogram(self, file_path, max_len, fs=32000, n_mels=128):
        y, _ = librosa.load(file_path, sr=fs)
        melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
    
        # padding to max_len
        current_len = melspec_db.shape[1]
        if current_len < max_len:
            pad_width = max_len - current_len
            melspec_db = np.pad(melspec_db, ((0, 0), (0, pad_width)), mode='constant')
        
        return melspec_db.flatten()
    
    def extract_trimmed_melspectrogram(self, file_path, trim_len, fs=32000, n_mels=128):
        y, _ = librosa.load(file_path, sr=fs)
        melspec = librosa.feature.melspectrogram(y=y, sr=fs, n_mels=n_mels)
        melspec_db = librosa.power_to_db(melspec, ref=np.max)
    
        # Trim to min_len
        melspec_db = melspec_db[:, :trim_len]
        return melspec_db.flatten()
    
    def select_representative_cluster_padded(self, file_paths, k=10, pca_dim=50):
        max_len = self.get_max_mel_length(file_paths)
        features = []
        for fp in tqdm(file_paths, desc='Extracting features'):
            feature = self.extract_padded_melspectrogram(fp, max_len=max_len)
            features.append(feature)
    
        features = np.array(features)
    
        # PCA
        pca = PCA(n_components=pca_dim, random_state=42)
        reduced_features = pca.fit_transform(features)

        # Exception
        k = min(k, len(file_paths))
    
        # KMeans clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(reduced_features)
    
        # Select representatives
        reps = []
        for cluster_id in range(k):
            indices = np.where(labels == cluster_id)[0]
            centroid = kmeans.cluster_centers_[cluster_id]
            distances = np.linalg.norm(reduced_features[indices] - centroid, axis=1)
            representative_idx = indices[np.argmin(distances)]
            reps.append(file_paths[representative_idx])
        
        return reps
    
    def select_representative_cluster_trimmed(self, file_paths, k=10, pca_dim=50):
        min_len = self.get_min_mel_length(file_paths)
        features = []
        # for fp in tqdm(file_paths, desc='Extracting trimmed features'):
        for fp in file_paths:
            feature = self.extract_trimmed_melspectrogram(fp, trim_len=min_len)
            features.append(feature)
    
        features = np.array(features)
    
        # PCA
        pca = PCA(n_components=0.95, random_state=42)
        reduced_features = pca.fit_transform(features)

        # Exception
        k = min(k, len(file_paths))
    
        # KMeans clustering
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(reduced_features)
    
        # Select representatives
        reps = []
        for cluster_id in range(k):
            indices = np.where(labels == cluster_id)[0]
            centroid = kmeans.cluster_centers_[cluster_id]
            distances = np.linalg.norm(reduced_features[indices] - centroid, axis=1)
            representative_idx = indices[np.argmin(distances)]
            reps.append(file_paths[representative_idx])
        
        return reps
    
    def cut_audio(self, y):
        num_samples = self.config.SR * self.config.DURATION
        if y.shape[0] < num_samples:
            # 오디오 길이가 5초 보다 짧으면 중앙(2~3초)에 패딩 추가
            # 패딩 방법 (위키독스 참고) : https://wikidocs.net/193732
            pad_size = num_samples - y.shape[0]
            left_pad = pad_size // 2
            right_pad = pad_size - left_pad
            y = np.pad(y, (0, 0), (left_pad, right_pad), mode='constant')
            
        else:
            # # 처음 5초만 사용
            # y = y[:, :num_samples]

            # 무작위 5초 클립 사용
            max_start_sample = int(y.shape[0] - num_samples)
            start_sample = int(random.uniform(0, max_start_sample)) # 무작위 시작점 설정
            y = y[start_sample: start_sample + num_samples] # 무작위 5초 클립

        return y

    def lower_upper_freq(self, y):
        r = torch.randint(self.config.N_MELS//2, self.config.N_MELS, size=(1,))[0].item()
        x = (torch.rand(size=(1,))[0]/2).item()
        
        # 여기가 문제인 부분입니다. 배열 리스트를 np.concatenate에 전달해야 합니다.
        arr1 = 1 - np.arange(r) * x / r
        arr2 = np.zeros(self.config.N_MELS - r) - x + 1
        
        pink_noise = (
            torch.from_numpy(
                np.array(
                    [
                        np.concatenate([arr1, arr2])  # 배열을 리스트로 감싸서 전달
                    ]
                )
            ).T
        ).float().to(self.config.DEVICE)
        
        y = y * pink_noise
        return y
    
    # def lower_upper_freq(self, y):
    #     r = torch.randint(self.config.N_MELS//2, self.config.N_MELS, size=(1,))[0].item()
    #     x = (torch.rand(size=(1,))[0]/2).item()
    #     pink_noise = (
    #         torch.from_numpy(
    #             np.array(
    #                 [
    #                     np.concatenate(
    #                         1 - np.arange(r) * x / r,
    #                         np.zeros(self.config.N_MELS - r) - x + 1,
    #                     )
    #                 ]
    #             )
    #         ).T
    #     ).float().to(self.config.DEVICE)

    #     y = y * pink_noise
    #     return y
        
    # log mel 사용
    def prepare_melspec(self, folder_path, folder_name, is_combined=False):
        
        save_dir = os.path.join('/kaggle/working/train_audio_process', folder_name)
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
                
        if is_combined:
            combined = np.array([])

            # Select representative cluster
            file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.ogg')]
            reps = self.select_representative_cluster_trimmed(file_paths, k=5)

            # Detect voice
            model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=True)

            for audio_path in reps:
                y, sr = librosa.load(audio_path, sr=self.config.SR)

                # Audio process
                y = self.process_audio(y, fs=self.config.SR, model=model, utils=utils)
                
                nonsilent_ranges = self.detect_nonsilent_custom(
                    y, sr=self.config.SR, min_silence_len=300, silence_thresh=-45
                )

                y = np.concatenate([
                    y[int(start * sr / 1000):int(end * sr / 1000)]
                    for start, end in nonsilent_ranges
                ])
                
                combined = np.concatenate((combined, y), axis=0)
                    
            combined = self.cut_audio(combined)

            # Background noise
            combined = self.am_audio_transforms(combined, sample_rate=self.config.SR)
            combined = self.np_audio_transforms(combined)
            
            # combined = self.audio_transforms(combined, sample_rate=self.config.SR)

            # sound_file_path = f'/kaggle/working/{folder_name}.ogg'
            # soundfile.write(sound_file_path, combined, self.config.SR)

            # 여기서 NumPy 배열을 PyTorch 텐서로 변환
            combined = torch.from_numpy(combined).float().unsqueeze(0)  # [1, time]
            if self.config.DEVICE == 'cuda':
                combined = combined.cuda()

            combined = self.melspec_transform(combined)
            combined = self.db_transform(combined)

            # =======================
            # get_spectrogram spec * 4 -> stack (추후 개선)
            # =======================

            if self.config.NORMAL == 80:
                combined = (combined + 80) / 80
            elif self.config.NORMAL == 255:
                combined = combined / 255
            else:
                raise NotImplementedError

            if self.mode == 'train':
                combined = self.time_mask_transform(combined)
                if torch.rand(size=(1,))[0] < 0.5:
                    combined = self.freq_mask_transform(combined)
                if torch.rand(size=(1,))[0] < 0.5:
                    combined = self.lower_upper_freq(combined)

            # PyTorch 텐서를 NumPy로 변환하여 저장
            combined_np = combined.cpu().numpy()
            
            # Save as NPY                
            file_name = folder_name + '.npy'
            npy_file_path = os.path.join(save_dir, file_name)
            np.save(npy_file_path, combined_np)       
            
            return combined
        
        else:
            mel_spec_list = []

            # Select representative cluster
            file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.ogg')]
            reps = self.select_representative_cluster_trimmed(file_paths, k=5)

            # Detect voice
            model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=True)
            
            for audio_path in reps:
                y, sr = librosa.load(audio_path, sr=self.config.SR)

                # Audio process
                y = self.process_audio(y, fs=self.config.SR, model=model, utils=utils)
                
                nonsilent_ranges = self.detect_nonsilent_custom(
                    y, sr=self.config.SR, min_silence_len=300, silence_thresh=-45
                )

                y = np.concatenate([
                    y[int(start * sr / 1000):int(end * sr / 1000)]
                    for start, end in nonsilent_ranges
                ])
                
                y = self.cut_audio(y)

                # Background Noise
                y = self.am_audio_transforms(y, sample_rate=self.config.SR)
                y = self.np_audio_transforms(y)
                
                # y_tensor = self.audio_transforms(y_tensor, sample_rate=self.config.SR)
                    
                y = self.melspec_transform(y)
                y = self.db_transform(y)
    
                if self.config.NORMAL == 80:
                    y = (y + 80) / 80
                elif self.config.NORMAL == 255:
                    y = y / 255
                else:
                    raise NotImplementedError
    
                if self.mode == 'train':
                    y = self.time_mask_transform(y)
                    if torch.rand(size=(1,))[0] < 0.5:
                        y = self.freq_mask_transform(y)
                    if torch.rand(size=(1,))[0] < 0.5:
                        y = self.lower_upper_freq(y)
                
                mel_spec_list.append(y)

            return mel_spec_list

    def prepare_target(self, primary_label, secondary_labels, add_secondary_labels=False):
        if isinstance(secondary_labels, str):
            secondary_labels = eval(secondary_labels) # object -> list
            
        target = np.zeros(self.num_classes, dtype=np.float32)

        if primary_label != 'nocall': # nocall == No species
            primary_label = self.bird2id[primary_label]
            target[primary_label] = 1.0
            if add_secondary_labels:
                for s in secondary_labels:
                    if s!="" and s in self.bird2id:
                        target[self.bird2id[s]] = self.config.SECONDARY_COEF # 보조 라벨의 기여도 0.5
                
        # target = torch.from_numpy(target).float() # Numpy -> Tensor
        
        return torch.tensor(target, dtype=torch.float32)
    
    def __getitem__(self, idx):
        filename = self.df['filename'].iloc[idx]
        primary_label = self.df['primary_label'].iloc[idx]
        secondary_labels = self.df['secondary_labels'].iloc[idx]
        rating = self.df['rating'].iloc[idx]

        folder_name = filename.split('/')[0]
        folder_path = f'{self.train_audio}/{folder_name}'
        print(folder_path)

        audio_paths = list(Path(folder_path).glob('*.ogg'))

        is_combined = len(audio_paths) > 1 # is_combined = True
        
        mel_spec = self.prepare_melspec(folder_path, folder_name, is_combined)
        
        target = self.prepare_target(primary_label, secondary_labels)
        
        # if isinstance(mel_spec, list):
        #     print('a')
        #     mel_spec = np.array(mel_spec, dtypes=np.float32)
        
        # if self.transform is not None:
        #     res = self.transform(image=mel_spec)
        #     mel_spec = res['image'].astype(np.float32)
        # else:
        #     if isinstance(mel_spec, list):
        #         mel_spec = np.array(mel_spec, dtype=np.float32)

        # mel_spec = torch.tensor(mel_spec, dtype=torch.float32)

        # mel_spec = mel_spec.expand(3, -1, -1)# .permute(1, 2, 0).transpose(2, 0, 1)

        if isinstance(mel_spec, torch.Tensor):
            if self.transform is not None:
                mel_spec_np = mel_spec.cpu().numpy()
                res = self.transform(image=mel_spec_np)
                mel_spec = torch.tensor(res['image'], dtype=torch.float32)
        else:
            if isinstance(mel_spec, list):
                print('a')
                mel_spec = np.array(mel_spec, dtype=np.float32)
            
            if self.transform is not None:
                res = self.transform(image=mel_spec)
                mel_spec = res['image'].astype(np.float32)
            
            mel_spec = torch.tensor(mel_spec, dtype=torch.float32)
        
        # 차원 확인 및 3채널로 확장
        if len(mel_spec.shape) == 2:
            mel_spec = mel_spec.unsqueeze(0)
        if mel_spec.shape[0] == 1:
            mel_spec = mel_spec.expand(3, -1, -1)
            
        return {
            'mel_spec': mel_spec,
            'target': target,
            'rating': rating
        }

In [66]:
def get_optimizer(model, config):
    if config.OPTIMIZER == 'Adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr = config.LR,
            weight_decay = config.WEIGHT_DECAY
        )
    elif config.OPTIMIZER == 'AdamW':  
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr = config.LR,
            weight_decay = config.WEIGHT_DECAY
        )
    else:
        raise NotImplementedError(f'Optimizer {config.OPTIMIZER} not implemented')
    
    return optimizer

def get_scheduler(optimizer, config):
    if config.SCHEDULER == 'CosineAnnealingLR':
        scheduler = torch.optim.lr_scheduler.CosineAnealingLR(
            optimizer,
            T_max=config.T_MAX,
            eta_min=config.MIN_LR
        )
    elif config.SCHEDULER == 'ReduceLROnPlateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode = 'min',
            factor = 0.5,
            patience = 2,
            min_lr = config.MIN_LR,
            verbose = True
        )
    elif config.SCHEDULER == 'StepLR':
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size = config.EPOCHS // 3,
            gamma = 0.5
        )
    else:
        scheduler = None
        
    return scheduler

def get_criterion(config):
    if config.CRITERION == 'BCEWithLogitsLoss':
        criterion = nn.BCEWithLogitsLoss()
    else:
        raise NotImplementedError(f'Criterion {config.CRITERION} not implemented')

    return criterion

def mixup(x, y, alpha=1.0, device='cuda'):
    if alpha > 0:
        lam = torch.distributions.Beta(alpha, alpha).sample().item()
    else:
        lam = 1.0

    indices = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[indices]
    mixed_y = lam * y + (1 - lam) * y[indices]

    return mixed_x, mixed_y

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    if not pandas.api.types.is_numeric_dtype(submission.values):
        bad_dtypes = {x: submission[x].dtype for x in submission.columns if not pandas.api.types.is_numeric_dtype(submission[x])}
        raise ParticipantVisibleError(f'Invalid submission data types found: {bad_dtypes}')

    solution_sums = solution.sum(axis=0)
    scored_columns = list(solution_names[solution_sums > 0].index.values)
    assert len(scored_columns) > 0

    return kaggle_metric_utilities.safe_call_score(sklearn.metrics.roc_auc_score, solution[scored_columns].values, submission[score_columns].values, average='macro')

def train_one_epoch(model, dataloader, optimizer, criterion, config, scheduler=None):
    model.train()
    bar = tqdm(dataloader, desc='Training', total=len(dataloader))

    all_outputs = []
    all_losses = []
    all_targets = []

    for batch in bar:

        # 초기화 단계, 이번 스탭에서 쌓인 그래디언트를 0으로 초기화.
        optimizer.zero_grad()

        target = batch['target'].to(config.DEVICE)
        mel_spec = batch['mel_spec']
        
        if isinstance(mel_spec, list):
            batch_outputs = []
            batch_losses = []

            for i in range(len(mel_spec)):
                mel = mel_spec[i].unsqueeze(0).to(config.DEVICE)
                tgt = target[i].unsqueeze(0).to(config.DEVICE)

                # Mixup (데이터 혼합) - 사용하는 이유 : 목표하는 과제는 다양한 종류의 조류, 곤충 등의 사운드가 섞여 있음. -> 일반화
                x_mel, y_target = mixup(mel, tgt, alpha=config.ALPHA, device=config.DEVICE)
                outputs = model(x_mel)
                loss = criterion(outputs, y_target)

                batch_outputs.append(outputs)
                batch_losses.append(loss)

            outputs = torch.cat(batch_outputs)
            avg_loss = torch.mean(torch.stack(batch_losses))
        else:
            mel_spec = mel_spec.to(config.DEVICE)
            x_mel, y_target = mixup(mel_spec, target, alpha=config.ALPHA, device=config.DEVICE)
            outputs = model(x_mel)
            avg_loss = criterion(outputs, y_target)

        # 역전파 단계, 파라미터들의 에러에 대한 변화도를 계산하고 누적함.
        avg_loss.backward()
        
        # 최적화 단계, loss를 최소화 할 수 있도록 파라미터 업데이트.
        optimizer.step()

        all_outputs.append(outputs.detach().cpu())
        all_losses.append(avg_loss.detach().cpu())
        all_targets.append(target.detach().cpu())

        bar.set_postfix(loss=avg_loss.item(), lr=optimizer.param_groups[0]['lr'])

    if scheduler is not None:
        if isinstance(scheduler, config.SCHEDULER):
            scheduler.step(torch.mean(torch.stack(all_losses)))
        else:
            scheduler.step()

    final_loss = torch.mean(torch.stack(all_losses))
    train_score = metric(torch.cat(all_outputs), torch.cat(all_targets))

    return final_loss, train_score
    
def validate(model, dataloader, optimizer, criterion, metric, config):
    model.eval()
    bar = tqdm(dataloader, desc='Validation', total=len(dataloader))

    all_outputs = []
    all_losses = []
    all_targets = []

    with torch.no_grad():
        for batch in bar:
            target = batch['target'].to(config.DEVICE)
            mel_spec = batch['mel_spec']

            if isinstance(mel_spec, list):
                batch_outputs = []
                batch_losses = []

                for i in range(len(mel_spec)):
                    mel = mel_spec[i].unsqueeze(0).to(config.DEVICE)
                    tgt = target[i].unsqueeze(0).to(config.DEVICE)

                    outputs = model(mel)
                    loss = criterion(outputs, tgt)

                    batch_outputs.append(outputs)
                    batch_losses.append(loss)

                outputs = torch.cat(batch_outputs)
                avg_loss = torch.mean(torch.stack(batch_losses))
            else:
                mel_spec = mel_spec.to(config.DEVICE)
                outputs = model(mel_spec)
                avg_loss = criterion(outputs, target)

            all_outputs.append(outputs.detach().cpu())
            all_losses.append(avg_loss.detach().cpu())
            all_targets.append(target.detach().cpu())

            bar.set_postfix(val_loss=avg_loss.item())

    final_loss = torch.mean(torch.stack(all_losses))
    validation_score = metric(torch.cat(all_outputs), torch.cat(all_targets))

    bar.set_postfix(val_loss=final_loss.item(), score=validation_score)

    return final_loss, validation_score, all_outputs

In [67]:
def main(config):
    df = pd.read_csv(config.train)
    sub = pd.read_csv(config.submission)
    skf = KFold(n_splits=config.N_FOLD, shuffle=True, random_state=config.SEED)
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['primary_label'])):
        train_df = df.iloc[train_idx].reset_index(drop=True)
        valid_df = df.iloc[val_idx].reset_index(drop=True)
        # print(train_df.shape, train_df.head())
        # print(valid_df.shape, valid_df.head())

        A_transform = A.Compose([A.Normalize()])
        
        # dataset
        train_dataset = BirdDataset(train_df, config, A_transform, mode='train')
        valid_dataset = BirdDataset(valid_df, config, A_transform, mode='valid')

        # dataloader
        train_loader = DataLoader(
            train_dataset,
            batch_size = config.BATCH_SIZE,
            shuffle = True,
            num_workers = config.NUM_WORKERS,
            pin_memory = True,
            collate_fn = collate_fn,
            drop_last = True
        )

        valid_loader = DataLoader(
            valid_dataset,
            batch_size = config.BATCH_SIZE,
            shuffle = False,
            num_workers = config.NUM_WORKERS,
            pin_memory = True,
            collate_fn = collate_fn
        )

        # model
        num_classes = len(sub.columns.tolist()[1:])
        
        model = BirdCLEFCNN(config, num_classes)

        # print(model)

        optimizer = get_optimizer(model, config)
        criterion = get_criterion(config)
        scheduler = None
        
        # train
        final_loss = train_one_epoch(model, train_loader, optimizer, criterion, config, scheduler)
        print(final_loss)
        
        # validation
        final_loss, validation_score, all_outputs = validate(model, valid_loader, optimizer, criterion, metric, config)
        print(final_loss)
        print(validation_score)
        print(all_ouptuts)

In [68]:
config = Config()

In [69]:
def collate_fn(batch):
    """Custom collate function to handle different sized spectrograms"""
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return {}
        
    result = {key: [] for key in batch[0].keys()}
    
    for item in batch:
        for key, value in item.items():
            result[key].append(value)
    
    for key in result:
        if key == 'target' and isinstance(result[key][0], torch.Tensor):
            result[key] = torch.stack(result[key])
        elif key == 'mel_spec' and isinstance(result[key][0], torch.Tensor):
            shapes = [t.shape for t in result[key]]
            if len(set(str(s) for s in shapes)) == 1:
                result[key] = torch.stack(result[key])
    
    return result

In [ ]:
main(config)

/usr/local/lib/python3.10/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = Shift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch_audiomentations/core/composition.py:42: FutureWarning: Transforms now expect an `output_type` argument th

/kaggle/input/birdclef-2025/train_audio/strfly1



Finding min mel length: 100%|██████████| 377/377 [00:42<00:00,  8.91it/s]

Extracting trimmed features: 100%|██████████| 377/377 [00:39<00:00,  9.51it/s]
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
/root/.cache/torch/hub/snakers4_silero-vad_master/src/silero_vad/utils_vad.py:277: UserWarning: Sampling rate is a multiply of 16000, casting to 16000 manually!
  warnings.warn('Sampling rate is a multiply of 16000, casting to 16000 manually!')
/usr/local/lib/python3.10/dist-packages/audiomentations/core/transforms_interface.py:108: UserWarning: Warning: input samples dtype is np.float64. Converting to np.float32
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/audiomentations/core/audio_loading_utils.py:37: UserWarning: /kaggle/input/cracklingfire/4211__dobroide__firecrackling.wav had to be resampled from 44100 Hz to 32000 Hz. This hurt execution time.
  warnings.warn(


/kaggle/input/birdclef-2025/train_audio/grekis



Finding min mel length: 100%|██████████| 990/990 [01:37<00:00, 10.19it/s]

Extracting trimmed features: 100%|██████████| 990/990 [01:30<00:00, 10.90it/s]
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
/root/.cache/torch/hub/snakers4_silero-vad_master/src/silero_vad/utils_vad.py:277: UserWarning: Sampling rate is a multiply of 16000, casting to 16000 manually!
  warnings.warn('Sampling rate is a multiply of 16000, casting to 16000 manually!')
/usr/local/lib/python3.10/dist-packages/audiomentations/core/transforms_interface.py:108: UserWarning: Warning: input samples dtype is np.float64. Converting to np.float32
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/audiomentations/core/audio_loading_utils.py:37: UserWarning: /kaggle/input/chirpingbird/742791__thesewak__multiple-birds-chirping-with-people-talking-in-background.wav had to be resampled from 44100 Hz to 32000 Hz. This hurt execution time.
  warnings.warn(
/usr/lo

/kaggle/input/birdclef-2025/train_audio/yehcar1


... (111 outputs left out)

/kaggle/input/birdclef-2025/train_audio/smbani



Finding min mel length: 100%|██████████| 287/287 [00:24<00:00, 11.90it/s]

Extracting trimmed features: 100%|██████████| 287/287 [00:24<00:00, 11.86it/s]
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
/root/.cache/torch/hub/snakers4_silero-vad_master/src/silero_vad/utils_vad.py:277: UserWarning: Sampling rate is a multiply of 16000, casting to 16000 manually!
  warnings.warn('Sampling rate is a multiply of 16000, casting to 16000 manually!')
/usr/local/lib/python3.10/dist-packages/audiomentations/core/transforms_interface.py:108: UserWarning: Warning: input samples dtype is np.float64. Converting to np.float32
  warnings.warn(


/kaggle/input/birdclef-2025/train_audio/roahaw



Finding min mel length: 100%|██████████| 709/709 [01:22<00:00,  8.65it/s]

Extracting trimmed features:  42%|████▏     | 295/709 [00:35<01:14,  5.54it/s]


In [ ]:
print('automatic end...')

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/train_audio_process', 'zip', '/kaggle/working/train_audio_process')

In [ ]:
# Config().SR

In [ ]:
# list(Path(Config().train_audio + '/whbman1/').glob('*.ogg'))[:4]

In [ ]:
# original_sound_file_path = list(Path(Config().train_audio + '/whbman1/').glob('*.ogg'))[0]

In [ ]:
# ipd.Audio(original_sound_file_path, rate=Config().SR)

In [ ]:
# prossed_sound_file_path = '/kaggle/working/whbman1.ogg'
# ipd.Audio(prossed_sound_file_path, rate=Config().SR)

```primary_label```: 종에 대한 코드

- 조류의 경우 eBird 코드 사용 (예: gretin1 → Great Tinamou)

- 조류가 아닐 경우 iNaturalist 택슨 ID 사용 (예: 24322 → Red Snouted Tree Frog)

```secondary_labels```: 기록자가 기록에 포함되도록 표시한 종 라벨 목록

```latitude & longitude```: 녹음이 촬영된 위치의 좌표, 일부 조류 종은 지역적으로 '방언'을 부를 수 있으므로 훈련 데이터에서 지리적 다양성을 찾고 싶을 수 있음.

```author```: 녹음을 제공한 사용자. ```Unknown``` - 이름이 제공되지 않은 경우.

```filename```: 연관된 오디오 파일의 이름.

```rating```: Xeno-canto 사용자가 제공한 1..5(1 - 낮은 품질, 5 - 높은 품질)의 값. 0은 평가가 없음을 의미. iNaturalist와 CSA는 품질 평가를 제공하지 않습니다.

```collection```: ```XC```, ```iNat``` 또는 ```CSA```, 녹음이 어떤 컬렉션에서 가져왔는지 나타냄. 파일 이름은 또한 컬렉션과 그 컬렉션 내의 ID를 참조.

In [ ]:
# train = pd.read_csv('/kaggle/input/birdclef-2025/train.csv')
# train.head()